# Perception Test

In [1]:
# ASA Imports
from asa.affect_model.belief import AffectModel  # noqa: I001
from asa.core.observers import Observers
from asa.core.representations import EKMAN6
from asa.perception.decode_keyword import EKMAN6_KEYWORDS, KeywordDecoder
from asa.perception.text_console import TextConsole
from asa.runtime import run_agent


# Notebook Specifics / Temporary Functions
#

import logging
from asa._tools.custom_logging import setup_logging
setup_logging(level="DEBUG")
log = logging.getLogger("asa.observer.debug")


from asa.core.observers import Event  # noqa: E402, I001
from asa.core.affect import AffectEvidence, AffectState, Utterance  # noqa: E402

def read_or_end(prompt: str) -> str:
    """Notebook stand-in for Ctrl-D — no frontend here can send a real EOF."""
    text = input(prompt)
    if text.strip() == ":q":
        raise EOFError
    return text

# def log_event(event: Event) -> None:
#     """Temporary stand-in for the recorder — narrows by type so the fields are checked."""
#     when = event.at.strftime("%H:%M:%S.%f")[:-3]

#     if isinstance(event, Utterance):
#         # log.debug("heard    %s  %r", when, event.text)
#         log.debug("heard    %s  %s  %r", when, event.id, event.text)
#     elif isinstance(event, AffectEvidence):
#         fired = {k: round(v, 2) for k, v in event.affect.values.items() if v}
#         log.debug("evidence %s  %-5s %-16s %s", when, event.target, event.source, fired)
#     elif isinstance(event, AffectState):
#         log.debug("state    %s  other=%s self=%s", when,
#                   dict(event.other.values), dict(event.self_.values))
#     else:
#         log.debug("event    %s  %s  %r", when, event.schema, event)

#     # Full dataclass
#     log.debug("%-12s %s", event.schema, event)

def log_event(event: Event) -> None:
    """Temporary stand-in for the recorder — schema, time, id, then the payload."""
    when = event.at.strftime("%H:%M:%S.%f")[:-3]

    if isinstance(event, Utterance):
        log.debug("%-12s %s  %s  %r", event.schema, when, event.id, event.text)
    elif isinstance(event, AffectEvidence):
        fired = {str(k): round(v, 2) for k, v in event.affect.values.items() if v}
        log.debug("%-12s %s  %s  %-5s %-16s %s",
                  event.schema, when, event.of_input, event.target, event.source, fired)
    elif isinstance(event, AffectState):
        log.debug("%-12s %s  other=%s self=%s",
                  event.schema, when, dict(event.other.values), dict(event.self_.values))
    else:
        log.debug("%-12s %s  %r", event.schema, when, event)

    # Full dataclass
    log.debug("%-12s %s", event.schema, event)
    

# Establish the source, decoder and a simple EKMAN6 keyword decoder
# source = TextConsole()
source = TextConsole(read=read_or_end)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)

# Affectmodel is not implemented and just reports a stub
model = AffectModel()

# Obervers is not implemented so just produce a debug log
observers = Observers()
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)



DEBUG: asa.observer.debug.log_event.line_53 - utterance/1  11:38:40.828  7b20f5b1d6bd  'i am happy'
DEBUG: asa.observer.debug.log_event.line_65 - utterance/1  Utterance(text='i am happy', source='input:text_console', intended=None, id='7b20f5b1d6bd', at=datetime.datetime(2026, 8, 7, 11, 38, 40, 828928, tzinfo=datetime.timezone.utc), schema='utterance/1')
DEBUG: asa.observer.debug.log_event.line_56 - evidence/1   11:38:40.828  7b20f5b1d6bd  other decoder:rule     {'happiness': 0.7}
DEBUG: asa.observer.debug.log_event.line_65 - evidence/1   AffectEvidence(target=<Target.OTHER: 'other'>, affect=AffectVector(representation='ekman6/1', values={<SixEmotions.ANGER: 'anger'>: 0.0, <SixEmotions.DISGUST: 'disgust'>: 0.0, <SixEmotions.FEAR: 'fear'>: 0.0, <SixEmotions.HAPPINESS: 'happiness'>: 0.7, <SixEmotions.SADNESS: 'sadness'>: 0.0, <SixEmotions.SURPRISE: 'surprise'>: 0.0}), confidence=None, source='decoder:rule', rationale='matched: happiness=happy', computed_from=None, of_input='7b20f5b1d6bd'